# 3D Rising Ball Free Surface

Demonstration example for setting up particle swarms with different material properties. This system consists of a dense, high viscosity sphere falling through a background lower density and viscosity fluid. [FSSA](https://doi.org/10.1016/j.pepi.2010.04.007) can be enabled for this simulation.

##### Expected velocity vrms results from these default toy settings
* `1.6621108609574737` fssa_enabled = False
* `1.6750725105720512` fssa_enabled = True

In [ ]:
from underworld import UWGeodynamics as GEO
from underworld import visualisation as vis

In [ ]:
u = GEO.UnitRegistry

In [ ]:
velocity = 1.0 * u.centimeter / u.hour
model_length = 2. * u.meter
model_height = 1. * u.meter
refViscosity = 1e6 * u.pascal * u.second
bodyforce = 200 * u.kilogram / u.metre**3 * 9.81 * u.meter / u.second**2

KL = model_height
Kt = KL / velocity
KM = bodyforce * KL**2 * Kt**2

GEO.scaling_coefficients["[length]"] = KL
GEO.scaling_coefficients["[time]"] = Kt
GEO.scaling_coefficients["[mass]"]= KM

In [ ]:
fssa_enabled = True
Model = GEO.Model(elementRes=(16, 16, 16), 
                  minCoord=(-1. * u.meter, -1. * u.meter, -50. * u.centimeter), 
                  maxCoord=(1. * u.meter, 1. * u.meter, 50. * u.centimeter),
                  gravity =(0.0, 0.0, -9.81 * u.meter / u.second**2) )

In [ ]:
Model.outputDir = "3D_RisingBall"

In [ ]:
heavyMaterial = Model.add_material(name="Heavy", shape=GEO.shapes.Layer3D(top=Model.top, bottom=Model.bottom))
lightMaterial = Model.add_material(name="Light", shape=GEO.shapes.Sphere(center=(0., 0., 20.*u.centimetre), radius=20. * u.centimetre))

In [ ]:
lightMaterial.density = 10 * u.kilogram / u.metre**3
heavyMaterial.density = 500 * u.kilogram / u.metre**3

lightMaterial.viscosity = GEO.ConstantViscosity(1e6 * u.pascal * u.second)
heavyMaterial.viscosity = GEO.ConstantViscosity(1e6 * u.pascal * u.second)

In [ ]:
Model.set_velocityBCs(left=[0, None, None],
                      right=[0, None, None],
                      top=[None, None, 0.],
                      bottom=[None, None, 0], 
                      front=[None, 0., None],
                      back=[None, 0., None])

In [ ]:
Model.freeSurface = True

In [ ]:
Fig = vis.Figure(resolution=(1200,600))
Fig.Surface(Model.mesh, Model.projMaterialField, cullface=False, opacity=0.5)
#Fig.window()

In [ ]:
Model.init_model(temperature=None)

In [ ]:
# By default the dt ~ 5minutes, wit fssa we set it to 10
if fssa_enabled == True:
    Model.fssa_factor = 1.0
    dt = 10*u.minutes
else:
    dt = None

In [ ]:
Model.run_for(nstep=2, dt=dt)

In [ ]:
expected_vrms_nofssa = 1.6621108609574737
import numpy as np
# is withing 1% of original run
assert np.isclose( Model.stokes_SLE.velocity_rms(), expected_vrms_nofssa, rtol=1e-2 )